# Conexão com o banco de dados `fiap`

Notebook utilitário para validar o acesso ao Postgres `fiap` (mesma conexão usada por `etl/db.py` e pela API em `app/api`) e explorar rapidamente o schema dimensional (`dim_*`, `fct_*`) antes de escrever queries mais elaboradas.

**Pré-requisito:** copie `.env.example` para `.env` na raiz do projeto e preencha `FIAP_DB_HOST`, `FIAP_DB_USER`, `FIAP_DB_PASSWORD` (e `FIAP_DB_PORT`/`FIAP_DB_NAME` se diferentes do padrão). O `.env` nunca é commitado.

## 1 · Setup — raiz do projeto e engine

In [ ]:
import sys
from pathlib import Path

import pandas as pd

def find_project_root(start: Path) -> Path:
    """Sobe a arvore ate achar a raiz do repo (marcada por .git ou etl/)."""
    for p in [start, *start.parents]:
        if (p / '.git').exists() or (p / 'etl').is_dir():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

`etl.db.get_engine()` lê as credenciais do `.env` (via `python-dotenv`) e monta a connection string `postgresql+psycopg2://...`. Se alguma variável obrigatória (`FIAP_DB_USER`, `FIAP_DB_PASSWORD`, `FIAP_DB_HOST`) estiver ausente, a célula abaixo falha com `KeyError` — é o sinal para revisar o `.env`.

In [ ]:
from etl.db import get_engine

engine = get_engine()
engine.url.render_as_string(hide_password=True)

## 2 · Teste de conexão

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    versao = conn.execute(text('SELECT version()')).scalar_one()
    banco, usuario = conn.execute(text('SELECT current_database(), current_user')).one()

print(f'Conectado em: {banco} (usuário {usuario})')
print(versao)

## 3 · Explorar o schema dimensional

Lista as tabelas do schema configurado em `FIAP_DW_SCHEMA` (padrão `dw`), onde vivem os `dim_*`/`fct_*` do modelo estrela.

In [ ]:
import os

DW_SCHEMA = os.getenv('FIAP_DW_SCHEMA', 'dw')

query_tabelas = text("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = :schema
    ORDER BY table_name
""")

with engine.connect() as conn:
    df_tabelas = pd.read_sql(query_tabelas, conn, params={'schema': DW_SCHEMA})

print(f'{len(df_tabelas)} tabelas/views em "{DW_SCHEMA}"')
df_tabelas

## 4 · Consulta de exemplo

Ajuste `TABELA_EXEMPLO` para uma das tabelas listadas acima e rode uma amostra.

In [ ]:
TABELA_EXEMPLO = df_tabelas['table_name'].iloc[0] if not df_tabelas.empty else None

if TABELA_EXEMPLO:
    query_amostra = text(f'SELECT * FROM "{DW_SCHEMA}"."{TABELA_EXEMPLO}" LIMIT 10')
    with engine.connect() as conn:
        df_amostra = pd.read_sql(query_amostra, conn)
    print(f'Amostra de {DW_SCHEMA}.{TABELA_EXEMPLO}')
    display(df_amostra)
else:
    print('Nenhuma tabela encontrada no schema — nada para amostrar.')